In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Lakehouse - Dim Courier SCD2") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,io.delta:delta-spark_2.12:3.1.0,mysql:mysql-connector-java:8.0.33") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

# Đọc bảng bronze trực tiếp qua path
df_bronze = spark.read.format("delta").load("s3a://lakehouse/warehouse/delivery_bronze")

df_bronze = df_bronze.withColumn("ds_str", F.lpad(F.col("ds").cast("string"), 4, "0")) \
                      .withColumn("event_date", F.to_date(F.concat(F.lit("2022-"), F.col("ds_str")), "yyyy-MMdd"))

courier_region = df_bronze.select("courier_id", "region_id", "aoi_id", "event_date").dropDuplicates()

window_spec = Window.partitionBy("courier_id").orderBy("event_date")

# --- SỬ DỤNG COALESCE VÀ TO_DATE ĐỂ XỬ LÝ NULL CHUẨN XÁC ---
df_scd2 = courier_region \
    .withColumn("prev_region", F.lag("region_id").over(window_spec)) \
    .filter((F.col("prev_region").isNull()) | (F.col("prev_region") != F.col("region_id"))) \
    .withColumn("effective_date", F.col("event_date")) \
    .withColumn("raw_end_date", F.lead("event_date").over(Window.partitionBy("courier_id").orderBy("event_date"))) \
    .withColumn("end_date", F.coalesce(F.col("raw_end_date"), F.to_date(F.lit("2099-12-31")))) \
    .withColumn("is_active", F.when(F.col("raw_end_date").isNull(), F.lit(True)).otherwise(F.lit(False))) \
    .select("courier_id", "region_id", "aoi_id", "effective_date", "end_date", "is_active")

delta_path = "s3a://lakehouse/warehouse/dim_courier_silver"
df_scd2.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(delta_path)

print("Đã lưu bảng SCD2 dim_courier_silver thành công!")
df_scd2.orderBy("courier_id", "effective_date").show(20, truncate=False)

Đã lưu bảng SCD2 dim_courier_silver thành công!
+----------+---------+------+--------------+----------+---------+
|courier_id|region_id|aoi_id|effective_date|end_date  |is_active|
+----------+---------+------+--------------+----------+---------+
|23        |164      |48886 |2022-05-01    |2099-12-31|true     |
|27        |102      |5564  |2022-05-01    |2099-12-31|true     |
|49        |146      |55124 |2022-05-01    |2099-12-31|true     |
|116       |151      |1328  |2022-09-28    |2099-12-31|true     |
|160       |146      |55450 |2022-05-01    |2099-12-31|true     |
|202       |140      |54578 |2022-10-21    |2099-12-31|true     |
|216       |146      |28788 |2022-05-01    |2099-12-31|true     |
|229       |111      |42052 |2022-05-01    |2099-12-31|true     |
|237       |146      |31261 |2022-07-01    |2099-12-31|true     |
|260       |111      |35811 |2022-06-20    |2099-12-31|true     |
|286       |95       |42290 |2022-05-01    |2099-12-31|true     |
|325       |137      |17843 